# Multimodal robotics agents: turning actions into tokens

> Agents in earlier lectures acted in a digital world: calling tools, searching, writing code. Actions and feedback all moved as strings. Those loops had a clear boundary, and the environment was under our control.
>
> This lecture puts the Agent into the **physical world**, facing a real robot arm. The core problem is one: how the model turns what it sees and the instruction it hears into concrete motions of the arm. We will walk through that process and implement its core component from scratch — a model that jointly handles images, instructions, and actions, called a vision-language-action model (VLA).

Start from a number: gripper opening, a continuous value in [0, 1], for example 0.4. The model cannot emit a string of such decimals directly.

First, cut the interval [0, 1] into 256 equal bins.

Second, see which bin 0.4 falls into. Each bin has width 1/256, so 0.4 has index ⌊0.4×256⌋≈102.

Third, the model no longer generates the decimal 0.4; it generates the integer index "bin 102". At use time, take the center of bin 102, 0.4004, as the reconstruction; the error is less than half a bin width.

Do this for every continuous quantity. Seven action dimensions become seven integers in 0..255, then concatenate into a token string like text. Discretizing continuous actions into "words" lets the model treat actions the way it treats language — that is the core idea of VLA: **action is also language**.

Giving an Agent a real body that acts in the physical world is called embodied intelligence. Its actions are displacements, rotation angles, and gripper opening; results come back as sensor readings. Once an action is wrong it can damage things, so a failed run cannot be replayed the way a digital task can. Another difficulty of the real world is data collection. Web image-text pairs number in the billions; robot trajectories (Open X-Embodiment) number about one million, three orders of magnitude fewer. So the model in this lecture first borrows knowledge learned from the web, then learns to control a robot.

The lecture has four steps: define embodied intelligence and the action space, implement the core of a VLA from scratch, build a collection and batching pipeline for robot data, and finally see how physical-world feedback lets the model keep improving. We start from the two basic notions, embodied intelligence and VLA.


## 1. Embodied intelligence and VLA

### What a VLA is: seeing, understanding, and acting in one set of parameters

This section lays the first foundation of the lecture: what a VLA is. Completing "put the cup on the plate" requires three kinds of information together: the image tells where the cup is, the instruction tells what to do, and the action tells the arm how to move. The three stages can be three independent modules, or they can live in one model.

The traditional approach splits the three into three independent modules: a vision module locates objects, a planning module computes a trajectory, a control module drives motors. Modules pass data through hand-designed interfaces. A change in task semantics, for example from "grasp the cup" to "push the plate to the corner of the table", forces a redesign of those interfaces.

The other approach lets one model read the image and the instruction together and emit actions directly, with no hand-designed information passing in between. That model is a vision-language-action model, abbreviated VLA (Vision-Language-Action). A VLA puts perception, understanding, and control into one set of parameters and learns them jointly; the cost is that convergence depends on a large amount of data.

A VLA faces a difficulty earlier lectures did not: how to represent actions. In earlier lectures the model emitted strings (tool names, arguments, code), and we wrote a parser to turn them into real calls. In robot tasks, actions are continuous numbers (displacement, rotation, gripper opening), and the model cannot emit a string of floats directly.

The VLA solution is to discretize continuous actions into action tokens, so actions and text share one vocabulary, and the training objective remains next-token prediction. The word Action in the RT-2 paper title refers to this "action is also language" design.

With that, the embodied loop holds end to end: read an image, understand an instruction, emit an action, observe again, every stage driven by the same parameters. Next we give a formal definition of embodied intelligence and a concrete representation of the action space.


This section answers two concrete questions as statements: what the embodied loop is, and what numbers represent one robot action. The first says how the model runs in the physical world; the second gives code a concrete action format.

Embodied intelligence places the Agent loop fully in the physical world: perceive the environment (camera images), understand an instruction (natural language), emit an action (motor commands), observe the result (a new image frame). The loop structure is the same as in earlier lectures; what changes is that inputs and outputs are physical signals rather than strings.

To let the model emit an action, we first specify how many numbers represent one action. For a mobile manipulator, one action is seven continuous quantities: end-effector displacement increments along x, y, z (Δpos_x, Δpos_y, Δpos_z), rotation increments about three axes (Δrot_x, Δrot_y, Δrot_z), and gripper opening (gripper, in [0, 1]).

The model cannot emit continuous decimals, so each continuous quantity is cut into 256 equal-width bins, each labeled by an integer in 0..255. Mapping a continuous value to an integer index is discretization; each small interval is a bin. The action string also begins with a terminate bit that marks the end of the action.

A hand calculation of one value makes discretization concrete. Gripper opening a = 0.4 lies in [0, 1]; each bin has width 1/256 ≈ 0.0039, and it falls in bin

idx = min(255, ⌊a / (1/256)⌋) = ⌊0.4 × 256⌋ ≈ 102.

Converting the index back to a continuous value takes the bin center: â = (102 + 0.5) / 256 ≈ 0.4004, and the error from the true 0.4 is within half a bin width.

The next figure shows the contrast in data scale.


In [ ]:
# scale gap between web data and robot data
import numpy as np
import matplotlib.pyplot as plt

labels = ["web pairs\n(billions)", "Open X-Embodiment\n(~1M episodes)"]
counts = [5e9, 1.0e6]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, counts, color=["#4C72B0", "#DD8452"])
ax.set_yscale("log")
ax.set_ylabel("scale (log)")
ax.set_title("web data vs robotics data")
for i, v in enumerate(counts):
    ax.text(i, v, f"{v:.1e}", ha="center", va="bottom")
plt.tight_layout()
plt.show()
print("web image-text pairs outnumber robot trajectories by about 3 orders of magnitude; that is the core motive for reusing a VLM.")


## 2. Vision-language-action models

This section turns the action space of the previous section into a runnable model. The previous section specified how actions are represented; this section does two things: make "action is another language" into a concrete token string, then start from a minimal VLA, implement it from scratch, and train it. In a real system this is the core of the robot controller: teaching a language model to emit actions.

RT-2 builds the action string as follows. Concatenate the 7 bin indices plus a terminate bit into a space-separated string, for example `"1 128 91 241 5 101 127 200"`. This string enters the training set together with the natural-language instruction. The model is trained in a question-answer format: `Q: what action should the robot take to [task]? A:`, and the answer is the action string.

The training objective remains next-token prediction. Action tokens and text tokens are predicted position by position in the same sequence; the model does not need a dedicated output channel for continuous numbers.

Where the action tokens come from is a design choice. RT-2's PaLI-X backbone already assigns an independent token to each integer from 0 to 1000, so bin indices bind directly to those integer tokens. The PaLM-E backbone does not have that, so it occupies 256 of the least-used existing tokens.

Occupying existing tokens is called symbol tuning: the model adds no parameters, and only rewrites the meaning of old tokens as action bins, which is relabeling a few symbols in the old vocabulary. The next hand calculation walks through turning one action into a token string.


### Hand calculation: turning one action into a token string

This section applies the discretization formula to one real action, computes the full token string by hand, then inverts it back to continuous values to see the error. Once this step is clear, the toy trajectory in the code below is straightforward.

First, how many numbers one action must describe. The pose of the arm's end effector in 3D needs 6 numbers for a full description: translation increments along x, y, z, plus rotation increments about the three axes, plus gripper opening, for 7 dimensions in total. ACTION_RANGE in the code below sets a range per dimension; we reuse those ranges and quantize one real action dimension by dimension.

Discretization is independent per dimension. Dimension $d$ has interval $[lo_d, hi_d]$, and the bin index of action value $a_d$ is

$$\text{idx}_d = \min\Big(255,\ \Big\lfloor \frac{a_d - lo_d}{hi_d - lo_d} \times 256 \Big\rfloor\Big)$$

The formula first normalizes the action value's position in the interval to [0, 1], then multiplies by 256 to see which bin it falls in, clipping at 255.

Take one move of the arm: translate −0.20 m in x, 0.15 m in y, no motion in z, no rotation about any axis, gripper opening 0.9. Written as a 7-D action that is

$$a = [-0.20,\ 0.15,\ 0,\ 0,\ 0,\ 0,\ 0.9]$$

Substituting dimension by dimension gives the normalized position and bin index:

| dim | meaning | range | normalized $t$ | $t \times 256$ | bin |
|:---|:---|:---|:---|:---|:---|
| 0 | x translation (m) | $[-0.25, 0.25]$ | $0.05/0.5 = 0.10$ | 25.6 | 25 |
| 1 | y translation (m) | $[-0.25, 0.25]$ | $0.40/0.5 = 0.80$ | 204.8 | 204 |
| 2 | z translation (m) | $[-0.25, 0.25]$ | $0.25/0.5 = 0.50$ | 128.0 | 128 |
| 3 | rotation about x (rad) | $[-0.60, 0.60]$ | $0.60/1.2 = 0.50$ | 128.0 | 128 |
| 4 | rotation about y (rad) | $[-0.60, 0.60]$ | $0.60/1.2 = 0.50$ | 128.0 | 128 |
| 5 | rotation about z (rad) | $[-0.60, 0.60]$ | $0.60/1.2 = 0.50$ | 128.0 | 128 |
| 6 | gripper opening | $[0, 1]$ | $0.90/1.0 = 0.90$ | 230.4 | 230 |

Walk through the y-translation dimension: $t = (0.15 - (-0.25)) / (0.25 - (-0.25)) = 0.40/0.50 = 0.80$, times 256 is 204.8, floored to 204. Prefixing the terminate bit 0, the token string of this action is

```text
0 25 204 128 128 128 128 230
```

Inversion is the reverse process, taking the center of each bin $\hat{a}_d = lo_d + \frac{\text{idx}_d + 0.5}{256}(hi_d - lo_d)$. For y translation: $\hat{a}_1 = -0.25 + \frac{204.5}{256} \times 0.5 \approx 0.1494$, about 0.0006 m from the true 0.15. The error on every dimension is at most half a bin width: about 0.00195 m on translation, about 0.00469 rad on rotation, about 0.00391 on the gripper.

Quantization has a cost. The model can express only 256 discrete levels and cannot emit arbitrarily fine values; that is a resolution limit. What it buys is unified modeling of actions and text in one vocabulary, with no separate regression head for continuous values. More bins mean denser levels, but a larger vocabulary and more compute; RT-2's choice of 256 is a compromise between resolution and scale.

The code below puts this action into a toy trajectory, checks the hand calculation, and measures round-trip error. Round-trip error means: discretize the action into bins, invert back, and see how far that is from the original continuous value.


In [ ]:
# RT-2-style action-token encode/decode: 7-D action × 256 bins
import numpy as np

ACTION_DIM = 7
NUM_BINS = 256
# per-dimension action range: first 3 translation (m), middle 3 rotation (rad), last gripper opening (0-1)
ACTION_RANGE = np.array([
    [-0.25, 0.25], [-0.25, 0.25], [-0.25, 0.25],
    [-0.60, 0.60], [-0.60, 0.60], [-0.60, 0.60],
    [0.0, 1.0],
], dtype=np.float64)


def discretize(action):
    """Map a 7-D continuous action to 7 bin indices (256 uniform bins per dimension)."""
    lo = ACTION_RANGE[:, 0]
    hi = ACTION_RANGE[:, 1]
    idx = np.floor((action - lo) / (hi - lo) * NUM_BINS)
    return np.clip(idx, 0, NUM_BINS - 1).astype(int)


def to_action_string(bins, terminate=0):
    """Concatenate a terminate bit and bin indices into an RT-2-style token string."""
    return " ".join([str(terminate)] + [str(int(b)) for b in bins])


def parse_action_string(s):
    """Parse a token string; return (terminate bit, bin-index array)."""
    nums = [int(x) for x in s.split()]
    return nums[0], np.array(nums[1:])


def detokenize(bins):
    """Invert bin indices to continuous values (bin centers)."""
    lo = ACTION_RANGE[:, 0]
    hi = ACTION_RANGE[:, 1]
    return lo + (bins + 0.5) / NUM_BINS * (hi - lo)


# toy action sequence: a grasp trajectory, approaching the target pose, gripper closing from open
rng = np.random.default_rng(42)
toy_actions = np.zeros((5, ACTION_DIM))
toy_actions[:, 0] = np.linspace(-0.2, 0.1, 5)
toy_actions[:, 1] = np.linspace(0.15, -0.05, 5)
toy_actions[:, 6] = np.linspace(0.9, 0.1, 5)

print("per-step action -> token string (terminate bit + 7 bin indices)")
for t in range(5):
    print(f"step {t}: {to_action_string(discretize(toy_actions[t]))}")

# round-trip check: decode back to continuous values; error should be less than one bin width
errors = []
for t in range(5):
    bins = discretize(toy_actions[t])
    recon = detokenize(bins)
    errors.append(np.abs(recon - toy_actions[t]).max())
print("max round-trip error per step:", [f"{e:.4f}" for e in errors])
bin_w = (ACTION_RANGE[:, 1] - ACTION_RANGE[:, 0]) / NUM_BINS
assert all(e < bin_w.max() for e in errors), "round-trip error should be less than the max bin width"
print("Key observation: round-trip error stays within a bin width, so encode/decode holds.")


**Experiment: uniform bins vs quantile bins.** The check above used RT-2-style uniform bins, splitting the interval equally between min and max. That has a weakness: it assumes action values are spread across the whole interval. In real data, most action values often crowd into a small range, with an occasional far-away extreme; that extreme is an outlier. One outlier can stretch the interval, widen every bin, and waste resolution. OpenVLA switches to quantile bins, so bin edges follow the density of the data. The code below compares bin widths of the two schemes on synthetic data that contains outliers.


In [ ]:
# quantile binning (OpenVLA) vs min-max (RT-2): robustness to outliers
import numpy as np

rng = np.random.default_rng(7)
normal = rng.normal(0.0, 0.05, size=200)
actions = np.concatenate([normal, [0.9]])


def minmax_bin_width(x, num_bins):
    """RT-2 style: bin width of a uniform split of [min, max]."""
    return (x.max() - x.min()) / num_bins


def quantile_bin_width(x, num_bins, lo=1, hi=99):
    """OpenVLA style: bin width of a split of the [1%, 99%] quantiles."""
    qlo, qhi = np.percentile(x, [lo, hi])
    return (qhi - qlo) / num_bins


w_minmax = minmax_bin_width(actions, 256)
w_quant = quantile_bin_width(actions, 256)
print(f"min-max coverage per bin: {w_minmax:.4f}")
print(f"quantile coverage per bin: {w_quant:.4f}")
print(f"effective resolution gain: {w_minmax / w_quant:.1f}x")
assert w_quant < w_minmax, "with outlier actions, quantile bins should be narrower"
print("Key observation: one outlier action stretches the min-max interval; quantiles are unaffected.")


The training representation of action tokens is complete; decoding still has a problem. When actually controlling a robot, the model predicts tokens one by one at inference. Ordinary QA can emit any natural-language token; a robot task cannot — if argmax lands on a text token, the action is illegal.

RT-2's method is: at decode time on robot tasks, mask the vocabulary and allow sampling only on action tokens. That is called an output constraint. It does not change training and takes effect only at inference: set logits of non-action positions to −inf; after softmax those positions have probability 0 and are never selected.


### Hand calculation: where argmax lands after text tokens are masked

This section uses a minimal example to show what masking does, step by step. The code then reproduces the same conclusion on a toy vocabulary of 512 tokens.

Suppose the vocabulary has only 4 tokens: 0 and 1 are action bins, 2 and 3 are text tokens. At one step the model emits logits

$$\text{logits} = [1.0,\ 2.5,\ 3.0,\ 0.5]$$

Without masking, argmax takes the largest entry and lands on 2. Position 2 is a text token, and it has the largest logit, so sampling directly would yield an illegal action.

Masking replaces non-action positions with $-\infty$:

$$\text{masked} = [1.0,\ 2.5,\ -\infty,\ -\infty]$$

In softmax, $e^{-\infty}=0$, so those two positions have output probability exactly zero:

$$p = \frac{[e^{1.0},\ e^{2.5},\ 0,\ 0]}{e^{1.0} + e^{2.5}} \approx [0.18,\ 0.82,\ 0,\ 0]$$

Now argmax lands on position 1, an action bin.

Model weights are never changed; only the range of logits considered at inference changes, so the method is an output constraint, not retraining. The code below reproduces the same fact on a 512-token toy vocabulary: without masking, argmax lands on a text token; after masking, it lands inside the action vocabulary.


In [ ]:
# action-token vocabulary masking at decode time (output constraint)
import numpy as np
import torch

VOCAB_SIZE = 512            # toy vocabulary: 0-255 action bins, 256-511 text tokens
ACTION_IDS = torch.arange(0, 256)

rng = np.random.default_rng(0)
logits = torch.tensor(rng.normal(0.0, 1.0, size=(VOCAB_SIZE,)))
logits[400] = 4.0           # make one text token have the highest logit


def unmasked_dist(logits):
    """No masking; softmax directly."""
    return torch.softmax(logits, dim=-1)


def masked_dist(logits, action_ids):
    """Keep only action tokens after masking: set other logits to -inf."""
    m = torch.full_like(logits, -float("inf"))
    m[action_ids] = logits[action_ids]
    return torch.softmax(m, dim=-1)


p_full = unmasked_dist(logits)
p_mask = masked_dist(logits, ACTION_IDS)
top_full = int(p_full.argmax())
top_mask = int(p_mask.argmax())

print(f"unmasked: argmax token {top_full} ({'text' if top_full >= 256 else 'action'})")
print(f"masked:   argmax token {top_mask} ({'text' if top_mask >= 256 else 'action'})")
print(f"total probability of non-action tokens after masking: {p_mask[256:].sum().item():.2e}")
assert p_mask[256:].sum().item() < 1e-6, "after masking, non-action token probability should be 0"
assert top_mask < 256, "after masking, sampling should land on an action token"
print("Key observation: masking forces decode to sample inside the action vocabulary, so robot-task output is legal.")


The previous sections specified how actions are represented and how output is kept legal. This section assembles those parts into a complete model: a tiny VLA implemented from scratch in torch. A real VLA is large; here the scale is a toy, but the structure matches one-to-one, and once it runs the real model is understandable.

The model is three components. The first is a vision encoder that maps image features to a vector. A real model is a vision tower fused from SigLIP and DINOv2; here two fully connected layers suffice.

The second is an instruction encoder that maps the instruction to a vector. A real model is a Llama language backbone; here it is token embedding plus mean pooling.

The third is an action decoder that emits a distribution over action bins. A real model emits a 256-way distribution; here it emits 7×256 logits, 7 for the action dimensions and 256 for the bins per dimension.

The image vector and the instruction vector are concatenated and fed to the action decoder. Cross-entropy is computed only on action tokens, matching OpenVLA's training objective. Concatenation is the step where the two modalities join; the next section computes that numerically by hand.


### How the three pieces become one model: a hand calculation of vector concatenation

This section walks through every numerical step of concatenation on one sample, to see how two modalities join in one vector. The code later checks the same process. Dimensions match the code: d_model=32.

One image frame is an 8×8 grayscale image, flattened to 64 numbers. The vision encoder is two fully connected layers: the first compresses 64-D to 32-D, the second stays at 32-D, so the image becomes a vector $v \in \mathbb{R}^{32}$.

The instruction "pick the red cup" tokenizes to 4 tokens; each token looks up a 32-D vector, and the 4 vectors are averaged to $t \in \mathbb{R}^{32}$. After concatenation $[v;\ t] \in \mathbb{R}^{64}$ — information from both modalities sits in one vector.

The action decoder reads this 64-D vector: a fully connected layer returns it to 32-D, then a linear map to $7 \times 256 = 1792$ outputs, reshaped to 7 rows by 256 columns. Row $j$ is the logits of action slot $j$; softmax on that row is the distribution over 256 bins for that dimension.

At training time, cross-entropy is computed only on the $7 \times 256$ action logits; text tokens produce no supervision, matching OpenVLA's training objective.

"Fusing in one model" means all components share one set of parameters; image features, instruction features, and the action head are updated by the same backpropagation. There is no hand-specified order of "perceive, then plan, then control"; the model learns how to allocate features from data: if training samples say "image brightness plus instruction id determine the target bin", the model learns that map.


In [ ]:
# mini VLA: vision encoder (MLP) + instruction encoding (Embedding) + action decoder
import torch
import torch.nn as nn


class MiniVLA(nn.Module):
    """A tiny vision-language-action fusion model.

    Input: image_feat, one 8x8 grayscale image (flattened to 64-D), text_ids instruction tokens.
    Output: logits[B, ACTION_DIM, NUM_BINS], a 256-way distribution per action slot.
    """

    def __init__(self, vis_dim=64, vocab_size=8, d_model=32):
        super().__init__()
        self.vis_enc = nn.Sequential(
            nn.Linear(vis_dim, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model),
        )
        self.text_emb = nn.Embedding(vocab_size, d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.ReLU(),
            nn.Linear(d_model, ACTION_DIM * NUM_BINS),
        )

    def forward(self, image_feat, text_ids):
        """Return logits for each action slot."""
        x = image_feat.view(image_feat.size(0), -1)  # [B, 64] flatten
        v = self.vis_enc(x)                          # [B, d_model]
        t = self.text_emb(text_ids).mean(dim=1)      # [B, d_model]
        h = torch.cat([v, t], dim=-1)                # [B, 2*d_model]
        logits = self.head(h).view(-1, ACTION_DIM, NUM_BINS)
        return logits


m = MiniVLA()
n_params = sum(p.numel() for p in m.parameters())
print("MiniVLA parameter count:", n_params)


**Experiment: training the mini VLA.** The previous cell defined the model; this section gives it a task it can learn. We synthesize a toy dataset: each 8×8 image has a brightness, each instruction has an id, and the target action bin is jointly determined by "brightness level + instruction id". The model must use both image and text to predict correctly. After 200 training steps, we look at the fraction of validation samples where all 7 action slots are predicted correctly.


In [ ]:
# synthetic toy task: the action is jointly determined by image brightness and the instruction; train mini VLA
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

NUM_TEXT = 4
N_SAMPLES = 256


def make_samples(n):
    """Generate (image, instruction, target action bins). The target is determined by coarse brightness and instruction id."""
    imgs = np.random.uniform(0.0, 1.0, size=(n, 8, 8)).astype(np.float32)
    texts = np.random.randint(0, NUM_TEXT, size=(n, 1))
    brightness = np.round(imgs.mean(axis=(1, 2)) * 4).astype(int)   # 0..4
    bins = np.stack([
        np.clip(brightness * 30 + texts[:, 0] * 35 + j * 3, 0, 255)
        for j in range(ACTION_DIM)
    ], axis=1).astype(np.int64)
    return (torch.from_numpy(imgs), torch.from_numpy(texts),
            torch.from_numpy(bins))


train = make_samples(N_SAMPLES)
val = make_samples(64)

model = MiniVLA(vocab_size=NUM_TEXT)
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
crit = nn.CrossEntropyLoss()


def evaluate(model, data):
    """Action-token accuracy: a step counts as correct only if all 7 slots are predicted correctly."""
    imgs, texts, bins = data
    with torch.no_grad():
        logits = model(imgs, texts)
        pred = logits.argmax(dim=-1)
    return (pred == bins).all(dim=1).float().mean().item()


losses, accs = [], []
for step in range(200):
    opt.zero_grad()
    imgs, texts, bins = train
    logits = model(imgs, texts)
    loss = crit(logits.reshape(-1, NUM_BINS), bins.reshape(-1))
    loss.backward()
    opt.step()
    losses.append(loss.item())
    if step % 25 == 0:
        accs.append(evaluate(model, val))

print(f"last-step loss = {losses[-1]:.3f}")
print(f"val all-slots action-token accuracy = {accs[-1]:.2%}")


**Experiment: training curves and sampling.** The previous section left two curves, loss and validation accuracy. This section plots them to see convergence, then samples a few action tokens from the validation set. Sampling matters because at inference the model does not always take the most probable token; it draws from the distribution, and we need to confirm that samples still land inside the action vocabulary and can be decoded into continuous actions.


In [ ]:
# training curves and action-token sampling
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(losses)
ax[0].set_xlabel("step")
ax[0].set_ylabel("loss")
ax[0].set_title("action-token cross entropy")
ax[1].plot(np.arange(0, 200, 25), accs, marker="o")
ax[1].set_xlabel("step")
ax[1].set_ylabel("val action accuracy")
ax[1].set_title("all-7-bins exact match")
plt.tight_layout()
plt.show()

# sample action tokens for three examples and concatenate into strings
model.eval()
imgs, texts, bins = val
with torch.no_grad():
    logits = model(imgs[:3], texts[:3])
    sampled = torch.distributions.Categorical(logits=logits).sample()
for k in range(3):
    pred = sampled[k].tolist()
    true = bins[k].tolist()
    print(f"sample {k}: pred {to_action_string(pred)}")
    print(f"          true {to_action_string(true)}")
print("Key observation: sampled tokens all lie in 0-255, i.e. the action vocabulary, and can be decoded into continuous actions.")


Training a VLA has a detail that ordinary training does not. The models above can learn the task by training directly, but a real model has to avoid a pitfall: training only on robot data makes the model forget knowledge it already had.

The model is first pretrained on a large web image-text corpus and learns common knowledge such as what a cup is and what a plate is. If fine-tuning continues only on robot data, gradients from the new data rewrite internal parameters layer by layer, overwrite representations accumulated on language tasks, and the model forgets that common knowledge. The phenomenon is called catastrophic forgetting.

RT-2's method is co-fine-tuning: fine-tune not only on robot data, but on robot trajectories mixed with the original web vision-language data, and raise the sampling weight of robot data in each batch. Language data reminds the model not to drop common knowledge; robot data teaches it actions; both are trained together.

The next toy experiment with a shared backbone reproduces the effect. The shared part is a text encoder; the language task reads a class label from it, the robot task reads an action from it. To make forgetting actually happen, the robot head is a single linear map with no nonlinearity in between — so the action task cannot solve itself in the head and must rewrite the shared text features, and robot gradients truly hit the place where language ability lives. The code first pretrains only the language task, then compares the two routes "train actions only" and "co-fine-tuning".


In [ ]:
# co-fine-tuning: shared text encoder + language head + action head; pretrain the language task first
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)


class TwoTask(nn.Module):
    """Shared text encoder, with a language head and an action head.

    The shared part maps 8 instruction words to a 3-D bottleneck; lang_head and action_head
    both read this feature linearly. The action head has no nonlinear layer, so the robot
    task must truly rewrite the shared features for forgetting to appear.
    """

    def __init__(self, vocab_size=8, k=3):
        super().__init__()
        self.text_emb = nn.Embedding(vocab_size, k)
        self.vis_proj = nn.Linear(1, k)
        self.lang_head = nn.Linear(k, 4)
        self.action_head = nn.Linear(k * 2, ACTION_DIM * NUM_BINS)

    def forward_lang(self, text_ids):
        """Language task: text -> class logits."""
        h = self.text_emb(text_ids).mean(dim=1)
        return self.lang_head(h)

    def forward_robot(self, text_ids, brightness):
        """Robot task: text + brightness -> action logits."""
        h = self.text_emb(text_ids).mean(dim=1)
        v = self.vis_proj(brightness)
        feat = torch.cat([h, v], dim=-1)
        return self.action_head(feat).view(-1, ACTION_DIM, NUM_BINS)


VOCAB8 = 8
web_ids = torch.arange(VOCAB8).view(-1, 1)
web_label = torch.tensor([i % 2 for i in range(VOCAB8)])


def make_robot_batch(n=64):
    """Robot samples: random text + coarse brightness; target bins determined by (brightness, text//2)."""
    texts = np.random.randint(0, VOCAB8, size=(n, 1))
    brightness = np.random.uniform(0.0, 1.0, size=(n, 1)).astype(np.float32)
    b_level = np.round(brightness[:, 0] * 4).astype(int)     # 0..4
    bins = np.stack([
        np.clip(b_level * 30 + (texts[:, 0] // 2) * 40 + j * 3, 0, 255)
        for j in range(ACTION_DIM)
    ], axis=1).astype(np.int64)
    return torch.from_numpy(texts), torch.from_numpy(brightness), torch.from_numpy(bins)


def lang_accuracy(model):
    """Language-task accuracy (fraction of 8 instructions classified correctly)."""
    with torch.no_grad():
        logits = model.forward_lang(web_ids)
    return (logits.argmax(dim=1) == web_label).float().mean().item()


def robot_accuracy(model):
    """Robot action accuracy (all 7 slots correct)."""
    texts, brightness, bins = make_robot_batch(128)
    with torch.no_grad():
        logits = model.forward_robot(texts, brightness)
    return (logits.argmax(dim=-1) == bins).all(dim=1).float().mean().item()


# pretrain: train only the language task until classification is correct
model = TwoTask()
opt = torch.optim.Adam(
    list(model.text_emb.parameters()) + list(model.lang_head.parameters()), lr=1e-2)
crit_lang = nn.CrossEntropyLoss()
for step in range(200):
    opt.zero_grad()
    logits = model.forward_lang(web_ids)
    loss = crit_lang(logits, web_label)
    loss.backward()
    opt.step()
print("language accuracy after pretraining:", lang_accuracy(model))


In [ ]:
# two fine-tuning comparisons: train actions only vs co-fine-tuning (actions + a little language)
import matplotlib.pyplot as plt

crit = nn.CrossEntropyLoss()

# (a) train on robot data only: freeze the language head; shared features are pulled by the action task
a = TwoTask()
a.load_state_dict(model.state_dict())
for p in a.lang_head.parameters():
    p.requires_grad = False
opt_a = torch.optim.Adam([p for p in a.parameters() if p.requires_grad], lr=3e-2)

# (b) co-fine-tuning: train robot data and language data together; do not freeze the language head
b = TwoTask()
b.load_state_dict(model.state_dict())
opt_b = torch.optim.Adam(b.parameters(), lr=3e-2)

lang_a, lang_b = [], []
robot_a, robot_b = [], []
for step in range(400):
    # (a) feed only robot data
    opt_a.zero_grad()
    texts, brightness, bins = make_robot_batch(64)
    loss = crit(a.forward_robot(texts, brightness).reshape(-1, NUM_BINS),
                bins.reshape(-1))
    loss.backward()
    opt_a.step()

    # (b) 45 robot samples + the full language set, trained together
    opt_b.zero_grad()
    texts, brightness, bins = make_robot_batch(45)
    loss_r = crit(b.forward_robot(texts, brightness).reshape(-1, NUM_BINS),
                  bins.reshape(-1))
    loss_l = crit(b.forward_lang(web_ids), web_label)
    (loss_r + 0.3 * loss_l).backward()
    opt_b.step()

    if step % 50 == 0:
        lang_a.append(lang_accuracy(a))
        lang_b.append(lang_accuracy(b))
        robot_a.append(robot_accuracy(a))
        robot_b.append(robot_accuracy(b))

steps = list(range(0, 400, 50))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(steps, lang_a, marker="o", label="robot-only")
ax[0].plot(steps, lang_b, marker="s", label="co-fine-tuning")
ax[0].set_xlabel("fine-tune step")
ax[0].set_ylabel("language accuracy")
ax[0].set_title("language retention")
ax[0].legend()
ax[1].plot(steps, robot_a, marker="o", label="robot-only")
ax[1].plot(steps, robot_b, marker="s", label="co-fine-tuning")
ax[1].set_xlabel("fine-tune step")
ax[1].set_ylabel("action accuracy")
ax[1].set_title("action accuracy")
ax[1].legend()
plt.tight_layout()
plt.show()
print(f"actions only: language {lang_a[-1]:.2%}, action {robot_a[-1]:.2%}")
print(f"co-fine: language {lang_b[-1]:.2%}, action {robot_b[-1]:.2%}")
print("Key observation: training actions only collapses language ability; co-fine-tuning keeps language ability, and the action task still converges.")


### The step at which catastrophic forgetting happens

The code above has already run both routes. This section explains the step at which forgetting happens, and how the two gradients pull against each other.

In TwoTask the language task and the robot task share the same text_emb. Language-task gradients want text features to keep "which word belongs to which class"; robot-task gradients want the features in a shape that "can compute 7 action bins". The two update directions often pull against each other.

When only robot data is trained, each backward pass sees only robot-task gradients. After tens of steps, shared features are rewritten into a shape useful for actions and harmful for language classification; language information is overwritten by gradients. That is catastrophic forgetting. We deliberately gave the action head only a linear map and no nonlinearity so the action task cannot solve itself in the head — it must rewrite shared features, or forgetting would not actually appear.

Co-fine-tuning puts both kinds of data in the same batch. Each update sees language loss and action loss together: language loss reminds shared features that "class information is still in use"; action loss pushes features toward "can control actions". The two gradients restrain each other and converge to a place that roughly satisfies both. RT-2 also raises the sampling weight of robot data in each batch so the action task learns faster.

In the code above, route (b) puts 45 robot samples and all 8 language samples in one batch, then multiplies language loss by weight 0.3; that is a toy version of the strategy. After the run, the language-accuracy contrast is the clearest: training actions only drops to 50% (chance on an 8-instruction binary task); co-fine-tuning stays at 100%.


## 3. Collecting and batching robot data

This section treats where the data comes from. The first two sections specified how the model emits actions, but the model needs data to train, and robot data cannot be downloaded the way web text can. This section covers how robot data is collected, and how raw recordings become training samples the model can use.

Robot data is collected by teleoperation. Teleoperation means an operator remotely controls the robot: wearing a VR headset, or dragging the arm by hand, to complete one task. Throughout, the system records camera frames, gripper state, and action commands together.

One complete recording of that kind is a trajectory, an episode. The structure of one trajectory is: a language instruction, plus a sequence of image frames, plus a sequence of 7-D actions.

Open X-Embodiment aggregates data accumulated this way by 70-plus labs into a unified format; after filtering there are about 1 million trajectories. The next cells synthesize one toy trajectory and show the conversion from a raw recording to training samples.


### What a trajectory looks like: frame-aligned instruction, image, and action

This section uses a toy trajectory to make "frame-aligned" concrete, and to show how one frame becomes one training sample.

Teleoperation is recording a human demonstration, with a control command on every frame. While the operator completes the task, the system samples at a fixed rate; each timestep stores one image, gripper state, and the 7-D action at that instant. The whole recording is one trajectory: one instruction, a sequence of image frames, a sequence of actions, aligned in time — frame $t$ corresponds to action $t$.

The toy trajectory below makes the alignment concrete. It has 16 frames; the instruction is "pick the red cup". The disk in frame $t$ sits at a different position, and action $t$ changes with it: gripper opening closes from 0.95 at frame 0 to 0.15 at frame 15, and displacement increments transition from the initial value to the target. Paired frame by frame, the model can learn "seeing this image, execute this action".

At training time each frame becomes an independent sample (image, instruction, action token string); a 16-frame trajectory yields 16 samples. Real data for one task usually has tens to hundreds of frames; Open X-Embodiment's about 1 million trajectories come from 70-plus labs accumulating records this way. The next two cells first synthesize this trajectory, then turn it into training samples.


In [ ]:
# synthesize one teleoperation trajectory: image frames + actions + instruction
import numpy as np

rng = np.random.default_rng(3)
N_FRAMES = 16


def draw_frame(step):
    """Draw one 8x8 image: a disk moving from the top-left toward the bottom-right."""
    img = np.zeros((8, 8))
    r = 1.5
    cx = 1.5 + step / (N_FRAMES - 1) * 5.0
    cy = 1.5 + step / (N_FRAMES - 1) * 5.0
    for y in range(8):
        for x in range(8):
            if (x - cx) ** 2 + (y - cy) ** 2 <= r * r:
                img[y, x] = 1.0
    return img


frames = np.stack([draw_frame(t) for t in range(N_FRAMES)])
actions = np.stack([
    np.linspace(-0.2, 0.1, N_FRAMES),
    np.linspace(0.2, -0.1, N_FRAMES),
    np.linspace(-0.1, 0.05, N_FRAMES),
    np.linspace(-0.3, 0.2, N_FRAMES),
    np.linspace(0.1, -0.05, N_FRAMES),
    np.linspace(-0.2, 0.15, N_FRAMES),
    np.linspace(0.95, 0.15, N_FRAMES),
], axis=1)
instruction = "pick the red cup"

episode = {"instruction": instruction, "frames": frames, "actions": actions}
print("episode structure:",
      {k: (v.shape if hasattr(v, "shape") else v) for k, v in episode.items()})
print("instruction:", instruction)


In [ ]:
# trajectory -> training samples: each frame is (image, instruction, action token string)
samples = []
for t in range(N_FRAMES):
    tok = to_action_string(discretize(episode["actions"][t]))
    samples.append({"frame": episode["frames"][t],
                    "instruction": instruction, "action_tokens": tok})

print("action token strings of the first 3 samples:")
for s in samples[:3]:
    print(" ", s["action_tokens"])
print("number of samples = number of frames =", len(samples))


Individual samples are ready; at training time they still have to be packed into a batch. In a batch, the length at the same position must be consistent. Action tokens are fixed at 8 integers, so they are already equal length; images are fixed-size arrays and tensorize directly. Only instruction text has variable length and needs special handling. The next section uses a toy batch to show the procedure, and points out a detail that is easy to miss.


### Only instructions need padding

This section explains why only instructions need padding. In a batch the three kinds of input have different length patterns. Action tokens are a fixed 8 integers (terminate bit plus 7 bin indices), so they are already equal length; images are fixed-size 8×8 arrays and stack directly into a tensor. The only variable length is instruction text: "pick the red cup" is 4 words, "pick the box" is 3 words.

The treatment is to pad each instruction to the longest in the batch, then attach an attention mask. Among the 5 instructions below the longest has 4 words; shorter sentences are padded with 0 at the end. The attention mask is the same length as the text: real token positions are 1, padding positions are 0, and the model looks only at positions where the mask is 1.

One detail is exactly what the toy data shows. In the code the id of "pick" is 0, and padding also fills 0; when the attention mask uses `pad_ids != 0`, the leading "pick" of every sentence is treated as padding as well — the first column of the printed mask is all 0. That does not affect the conclusion of this lecture's demo (the padding demo itself does not enter model training), but it shows that the padding id must differ from every real token; a real model uses a dedicated padding token id to avoid this clash.


In [ ]:
# instruction tokenization and batching (padding + attention mask)
import numpy as np

WORD2ID = {"pick": 0, "the": 1, "red": 2, "cup": 3, "blue": 4, "pen": 5,
           "box": 6, "bottle": 7}
instructions = [
    "pick the red cup", "pick the blue pen", "pick the box",
    "pick the red box", "pick the bottle",
]


def tokenize(text):
    """Split the instruction into words and map them to ids."""
    return [WORD2ID[w] for w in text.split()]


tok = [tokenize(t) for t in instructions]
max_len = max(len(x) for x in tok)
pad_ids = [x + [0] * (max_len - len(x)) for x in tok]
pad_ids = np.array(pad_ids, dtype=np.int64)
attention_mask = (pad_ids != 0).astype(np.int64)

print("padded id matrix:\n", pad_ids)
print("attention mask:\n", attention_mask)
print("batch shape:", pad_ids.shape)

# action tokens are a fixed 8 integers and need no padding; count action tokens produced by one trajectory
print("action tokens produced by one 16-frame trajectory:", N_FRAMES * (ACTION_DIM + 1))
print("Key observation: action tokens are equal length; only variable-length instruction text needs padding.")


## 4. Feedback from learning in the physical world

The previous two sections treated where data comes from and how it is processed. Data alone is not enough; after training the model still has to know whether it can actually do the work. This section covers the loop in which the model keeps improving in the physical world.

Training is not a one-shot event. A typical deployment loop is: collect a batch of trajectories by teleoperation → train the model → closed-loop evaluation on a real arm → correct failed trajectories and return them to the dataset → train again.

Closed-loop evaluation means letting the model work in real time: the arm's camera reads the current image, the model emits the next action from the image and the instruction, the robot executes it, then we look at the result. Every step happens in the real world, rather than replaying a dataset.

Many robots train first in simulation, which is cheaper. Simulation and reality differ: physics parameters, lighting, and textures in simulation are more idealized, and success in simulation does not guarantee success in the real environment. That gap is called the sim-to-real gap, so closed-loop evaluation has to return to the real robot.

Closed-loop evaluation also exposes data-quality problems. When OpenVLA cleaned Bridge data it found a large number of all-zero actions — the gripper did not move but was recorded as zero displacement, and such samples directly contaminate the action-token distribution. Data cleaning itself is part of "learning from the physical world": neither recordings nor labels are perfect, and feedback is needed to sort them. The code below uses a simplified model of the recycle process: failed samples are corrected and returned to the dataset, the data volume doubles, and evaluation success rises along a saturating curve.


In [ ]:
# simplified feedback loop: failed samples are corrected and recycled; data volume doubles; eval success rises along a saturating curve
import numpy as np
import matplotlib.pyplot as plt


def learning_curve(n, acc_max=0.92, k=600):
    """Saturating map from data volume to success rate: acc = acc_max * (1 - exp(-n/k))."""
    return acc_max * (1 - np.exp(-n / k))


rounds = np.arange(5)
n_data = 200 * (2 ** rounds)
success = learning_curve(n_data)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(rounds, success, marker="o")
ax.set_xlabel("collection round")
ax.set_ylabel("eval success rate")
ax.set_title("failure samples recycled into training")
for r, n, s in zip(rounds, n_data, success):
    ax.annotate(f"n={n}", (r, s), textcoords="offset points",
                xytext=(0, 8), ha="center")
plt.tight_layout()
plt.show()
print("cumulative trajectories:", n_data.tolist())
print("eval success rate:", [round(s, 3) for s in success])


A last question, stated as a design choice: how language reasoning connects to low-level control. The models above emit action tokens directly, but many tasks need thinking first, then acting. One RT-2 variant expands each example into a format such as "Instruction: I'm hungry. Plan: pick the snack. Action: 1 128 124 ...", so the model first generates a natural-language plan, then emits action tokens. The method borrows the idea of chain-of-thought, written CoT. The language plan carries reasoning ability from web pretraining; action tokens remain constrained to the action vocabulary. The next cells walk through this chained format with llm_client, then invert the action string back to a continuous action.


### Write the plan first, then the action

The previous section said the model generates a plan first, then emits an action. This section explains why that order works, and how the action part is kept legal.

Plan→Action expands one training sample from "instruction + action" into three segments: instruction, natural-language plan, action tokens. RT-2's CoT variant augments robot data with the template "Instruction: ... Plan: ... Action: ...".

The key is that the plan carries ability from web pretraining. Web corpora taught the model the chain "reason in language first, then give a conclusion", while robot trajectories have no reasoning process, only "which action to take". Augmentation makes the model write one sentence of natural-language reasoning before emitting an action, stating common knowledge such as "if hungry, pick a snack" explicitly. The language plan is the bridge from web knowledge to actions: the model first calls the language reasoning it already knows, then lands on the unfamiliar action tokens.

The action part is still under an output constraint. At inference, decoding the Action segment allows sampling only on action bins and the terminate bit, so the result can be inverted into a legal control quantity. The language part can use reasoning freely; the action part stays executable; the two play their roles in the same sequence.

The code below feeds the instruction to llm_client. When a live model interface is configured, the model emits Plan and Action in the template; when the interface is unavailable, the code falls back to rules and generates a parseable placeholder. Finally the Action string is inverted to a continuous action, checking the closed loop.


In [ ]:
# initialize the LLM client (without a key this enters the real-API demo)
import os
import sys

_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'llm_client.py')):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)

from llm_client import get_llm

client = get_llm()
print("LLM client:", "scripted example (placeholder output)" if False else "real API")


In [ ]:
# Plan + Action chain-of-thought: generate a plan, then emit action tokens
import re

# rule-based fallback planner: provides a parseable Plan/Action placeholder under the real-API demo
FALLBACK = {
    "hungry": ("pick the snack", "0 140 90 220 30 70 200 60"),
    "thirsty": ("pick the cup", "0 120 70 240 20 80 210 80"),
}


def rule_plan(instruction):
    """Return (plan, action string) by keyword; return None if none matches."""
    for key, value in FALLBACK.items():
        if key in instruction:
            return value
    return None


def parse_action_part(text):
    """Extract the 'Action: <space-separated numbers>' part from the reply; return None if missing."""
    m = re.search(r"Action:\s*([\d\s]+)", text)
    return m.group(1).strip() if m else None


instruction = "I am hungry, pick a snack and hand it to me"
prompt = (
    f"Instruction: {instruction}\n"
    "Plan: <one-sentence natural-language plan>\n"
    "Action: <terminate bit + 7 bin indices, space-separated>"
)

reply = client.chat([{"role": "user", "content": prompt}])
print("model reply:\n" + reply)

if parse_action_part(reply) is None:
    plan, action_str = rule_plan("hungry")
    source = "scripted demo using the rule fallback (placeholder)"
else:
    action_str = parse_action_part(reply)
    m = re.search(r"Plan:\s*(.+)", reply)
    plan = m.group(1).strip() if m else None
    source = "live LLM output"

print("\nsource:", source)
print("Plan:", plan)
print("Action:", action_str)
term, bins = parse_action_string(action_str)
recon = detokenize(bins)
print("reconstructed continuous action (first 3 dims, meters):", np.round(recon[:3], 3))
print("gripper opening:", round(float(recon[6]), 3))
assert len(bins) == ACTION_DIM, "number of action tokens should match action dimension"
assert int(term) in (0, 1), "terminate bit should be 0 or 1"
print("Key observation: the language plan and action tokens are generated in one sequence; action tokens invert back to continuous controls.")


## Summary

What this section covered:

- A VLA puts vision, language, and action in one model: image and instruction are encoded and concatenated into a vector; the action head maps the vector to discrete action tokens
- Actions must be discretized into tokens to use a language model's output channel: each of 7 action dimensions is split into 256 bins, encoded as a token string, and decoded back to continuous values
- Vocabulary masking at decode time restricts sampling to action bins, so the model does not emit text instead of an action
- The action head computes cross-entropy only on action slots; language and action share the earlier encoder layers
- Co-fine-tuning mixes language and action losses on the same batch, so training actions does not catastrophically forget language ability
- Teleoperation records a human demonstration as frame-aligned instruction, image, and action; each frame becomes one training sample
- Plan→Action lets the model do high-level planning in language first, then land on a concrete action, carrying reasoning ability from web pretraining


## Exercises

> You may ask an AI to explain the idea. Do not ask it to finish the exercise for you.

All three exercises reuse code from this lecture. Complete them on paper first, then run the assertions.


**Exercise 1: action-token discretization and inversion**

Complete `quantile_bins_per_dim` so each dimension's bin interval is set by the 1st–99th percentiles, and complete `quantile_discretize` and `quantile_detokenize` for a quantization round-trip. The assertions check: round-trip error stays within the quantile bin width, and quantile bins are clearly narrower than min-max bins.

Hint: take the 1st and 99th percentiles per dimension to set interval endpoints; clip out-of-range values.


In [ ]:
# blanks: quantile bin intervals and a quantization round-trip
def quantile_bins_per_dim(actions_2d, num_bins=256, lo=1, hi=99):
    """Return an array of quantile bin widths, one per dimension."""
    qlo = np.percentile(actions_2d, lo, axis=0)
    qhi = np.percentile(actions_2d, hi, axis=0)
    return (qhi - qlo) / num_bins


def quantile_discretize(action, bin_edges):
    """action: 7-D array; bin_edges: (lower-bound array, upper-bound array)."""
    lo, hi = bin_edges
    idx = np.floor((action - lo) / (hi - lo) * NUM_BINS)
    return np.clip(idx, 0, NUM_BINS - 1).astype(int)


def quantile_detokenize(bins, bin_edges):
    """Invert bin indices to bin-center values."""
    lo, hi = bin_edges
    return lo + (bins + 0.5) / NUM_BINS * (hi - lo)


# construct toy data that contains outlier actions
rng = np.random.default_rng(5)
normal = rng.normal(0.0, 0.05, size=(200, ACTION_DIM))
outliers = rng.uniform(0.7, 0.9, size=(2, ACTION_DIM))
data = np.concatenate([normal, outliers], axis=0)

w_minmax = (data.max(axis=0) - data.min(axis=0)) / 256
w_quant = quantile_bins_per_dim(data)
assert np.all(w_quant < w_minmax), "with outlier actions, quantile bins should be narrower"
print("min-max mean bin width:", np.mean(w_minmax))
print("quantile mean bin width:", np.mean(w_quant))

qlo = np.percentile(data, 1, axis=0)
qhi = np.percentile(data, 99, axis=0)
edges = (qlo, qhi)
for k in range(5):
    a = np.clip(data[k], qlo, qhi)
    recon = quantile_detokenize(quantile_discretize(a, edges), edges)
    err = np.abs(recon - a).max()
    assert err < w_quant[k], "round-trip error should stay within the quantile bin width"
print("Exercise 1 passed: quantile binning is robust to outliers; quantization round-trip error stays within a bin width.")


**Exercise 2: encoding and decoding action token strings**

Complete `string_to_bins` and `string_to_action` so an action string can be inverted to bin indices and reconstructed as a continuous action. The assertions check: the string is space-separated with 8 elements, and string-level round-trip error is less than the maximum bin width.

Hint: `str.split` separates the terminate bit and 7 bin indices; inversion reuses the bin-center formula.


In [ ]:
# blanks: string-level round-trip (self-contained parse)
def string_to_bins(s):
    """Parse an action string; return (terminate bit, 7 bin indices)."""
    nums = [int(x) for x in s.split()]
    return nums[0], np.array(nums[1:])


def string_to_action(s):
    """Parse an action string and invert it to a continuous action."""
    _, bins = string_to_bins(s)
    lo = ACTION_RANGE[:, 0]
    hi = ACTION_RANGE[:, 1]
    return lo + (bins + 0.5) / NUM_BINS * (hi - lo)


rng = np.random.default_rng(9)
acts = rng.uniform(ACTION_RANGE[:, 0], ACTION_RANGE[:, 1],
                   size=(8, ACTION_DIM))
max_err = 0.0
for a in acts:
    s = to_action_string(discretize(a))
    assert len(s.split()) == ACTION_DIM + 1, "terminate bit plus 7 bins is 8 integers"
    recon = string_to_action(s)
    max_err = max(max_err, float(np.abs(recon - a).max()))
assert max_err < (ACTION_RANGE[:, 1] - ACTION_RANGE[:, 0]).max() / NUM_BINS
print("Exercise 2 passed: action token strings invert to continuous actions with error less than the max bin width.")


**Exercise 3: action-token vocabulary masking at decode time**

Complete `mask_for_robot_task`: set logits of non-action tokens to −inf, keeping only action tokens. The assertions check: after masking, non-action logits are −inf, softmax probability of non-action tokens is 0, and argmax lands inside the action vocabulary.

Hint: build a template with `torch.full_like(logits, -inf)`, then index-assign the action-position logits back.


In [ ]:
# blanks: action-token vocabulary masking
import torch


def mask_for_robot_task(logits, action_ids):
    """Mask non-action tokens, keeping logits only at action_ids."""
    masked = torch.full_like(logits, -float("inf"))
    masked[action_ids] = logits[action_ids]
    return masked


V = 512
ACTION_IDS = torch.arange(256)
rng = np.random.default_rng(2)
logits = torch.tensor(rng.normal(0.0, 1.0, size=(V,)))
logits[500] = 5.0                                   # a text token takes the highest score

masked = mask_for_robot_task(logits, ACTION_IDS)
assert torch.isinf(masked[256:]).all(), "non-action positions should be -inf"
probs = torch.softmax(masked, dim=-1)
assert probs[256:].sum().item() < 1e-6, "non-action token probability should be 0"
assert int(masked.argmax()) < 256, "argmax should land inside the action vocabulary"
print("Exercise 3 passed: after masking, sampling stays inside the action vocabulary, so robot-task output is legal.")


## References

- Brohan et al., [RT-2: Vision-Language-Action Models Transfer Web Knowledge to Robotic Control](https://arxiv.org/abs/2307.15818), 2023 — founding VLA work: action as token, co-fine-tuning, transfer of web knowledge
- Kim et al., [OpenVLA: An Open-Source Vision-Language-Action Model](https://arxiv.org/abs/2406.09246), 2024 — the first open general-purpose VLA; a systems study of quantile binning, LoRA, and quantized inference
- Brohan et al., [RT-1: Robotics Transformer for Real-World Control at Scale](https://arxiv.org/abs/2212.06817), 2022 — a 35M-parameter discretized-action transformer, the foundation and data source of RT-2
- Open X-Embodiment Collaboration, [Open X-Embodiment: Robotic Learning Datasets and RT-X Models](https://arxiv.org/abs/2310.08864), 2023 — 70+ sub-datasets and cross-embodiment RT-X; OpenVLA's training data source
- Driess et al., [PaLM-E: An Embodied Multimodal Language Model](https://arxiv.org/abs/2303.03378), 2023 — an embodied multimodal language model, the VLM backbone of RT-2-PaLM-E
- Physical Intelligence, [π0: A Vision-Language-Action Flow Model](https://www.physicalintelligence.company), 2024 — an alternative route that replaces discrete tokens with a flow-matching continuous action head
- Karamcheti et al., [Prismatic VLMs](https://arxiv.org/abs/2402.07865), 2024 — OpenVLA's VLM backbone, a SigLIP+DINOv2 fused vision encoder
- Hu et al., [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685), 2021 — low-rank adaptation, the basis of OpenVLA's parameter-efficient fine-tuning
- Chi et al., [Diffusion Policy: Visuomotor Policy Learning via Action Diffusion](https://arxiv.org/abs/2303.04137), 2023 — a continuous-action diffusion policy, the baseline OpenVLA fine-tuning is compared against
- CS329A course outline [Lecture 16: Multimodal AI Agents in Robotics](https://cs329a.stanford.edu/) — this lecture's place on the course map
